# 04 - Module 1: mural LoRA for Stage 1 (no synthetic damage)
Fine-tunes a rank-8 LoRA on the **Stable Diffusion 1.5 inpainting** UNet using **all 5,096 clean MuralDH murals only**. No synthetic damage is added: for every mural the mask is the whole image, so the model learns to paint complete Dunhuang murals (style, pigments, plaster texture) from the trigger prompt `dunhuang_mural_style`.

**Settings:** Accelerator = **GPU T4**, Internet = **ON**, Input = `MuralDH`. About 40-50 min for 1000 steps.

Output: `/kaggle/working/lora_inpaint/dunhuang_inpaint_lora.safetensors` (keys carry the `unet.` prefix).


In [ ]:
# Kaggle ships torchao 0.10 which peft rejects; we don't use torchao, so switch that dispatcher off.
try:
    import peft.tuners.lora.torchao as _pt
    _pt.is_torchao_available = lambda: False
except Exception as e:
    print("torchao patch skipped:", e)

import os, re, json, math, random, time, gc
import numpy as np, cv2, torch, torch.nn.functional as F
from pathlib import Path
from PIL import Image
import diffusers, peft, transformers
print("torch", torch.__version__, "| diffusers", diffusers.__version__, "| peft", peft.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Turn on a GPU accelerator"

IMAGE_DIR = Path("/kaggle/input/datasets/balamsd07/muraldh/Mural512/Mural512")
OUT = Path("/kaggle/working/lora_inpaint"); OUT.mkdir(parents=True, exist_ok=True)
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-inpainting"
CAPTION = "dunhuang_mural_style, a Dunhuang Buddhist mural painting, mineral pigments, aged plaster wall"
SEED, STEPS, BATCH, ACCUM, LR, RANK = 42, 1000, 2, 2, 1e-4, 8
SAVE_EVERY = 250
dev = torch.device("cuda")

from huggingface_hub import model_info
print("model access:", model_info(MODEL_ID).id)

In [ ]:
# ---- all MuralDH crops are used for training (no held-out split, no synthetic damage) ----
pat = re.compile(r"img_(\d+)crop_(\d+)_(\d+)\.png$")
train_files = sorted(p for p in IMAGE_DIR.glob("*.png") if pat.match(p.name))
print("training murals:", len(train_files))


In [ ]:
class MuralDS(torch.utils.data.Dataset):
    """clean mural only. No damage: the 'mask' is the whole image."""
    def __init__(self, files): self.files = files
    def __len__(self): return 10**6
    def __getitem__(self, i):
        rng = np.random.default_rng((i * 2654435761 + int(time.time()*1000)) % (2**32))
        img = np.asarray(Image.open(self.files[int(rng.integers(len(self.files)))]).convert("RGB").resize((512, 512)))
        if rng.random() < 0.5: img = img[:, ::-1]
        x = torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1).float() / 127.5 - 1
        return dict(pixel=x, drop=bool(rng.random() < 0.1))

dl = torch.utils.data.DataLoader(MuralDS(train_files), batch_size=BATCH, num_workers=2, pin_memory=True)
b = next(iter(dl)); print(b["pixel"].shape)


In [ ]:
from diffusers import StableDiffusionInpaintPipeline, UNet2DConditionModel, AutoencoderKL, DDPMScheduler
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig
from diffusers.training_utils import cast_training_params

tok = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
text_enc = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder", torch_dtype=torch.float16).to(dev).eval()
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float16).to(dev).eval()
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet", torch_dtype=torch.float16).to(dev)
sched = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
print("UNet in_channels:", unet.config.in_channels, "(must be 9)"); assert unet.config.in_channels == 9

def enc(text):
    ids = tok(text, padding="max_length", max_length=tok.model_max_length, truncation=True, return_tensors="pt").input_ids.to(dev)
    with torch.no_grad(): return text_enc(ids)[0]
emb_cap, emb_null = enc(CAPTION), enc("")
del text_enc; gc.collect(); torch.cuda.empty_cache()

for m in (vae, unet): m.requires_grad_(False)
unet.add_adapter(LoraConfig(r=RANK, lora_alpha=RANK, init_lora_weights="gaussian", target_modules=["to_q", "to_k", "to_v", "to_out.0"]))
cast_training_params(unet, dtype=torch.float32)          # LoRA weights in fp32, frozen base in fp16
unet.enable_gradient_checkpointing()
params = [p for p in unet.parameters() if p.requires_grad]
print("trainable params:", sum(p.numel() for p in params))
opt = torch.optim.AdamW(params, lr=LR, weight_decay=1e-2)
scaler = torch.cuda.amp.GradScaler()

In [ ]:
from peft.utils import get_peft_model_state_dict
from diffusers.utils import convert_state_dict_to_diffusers

def save_lora(path):
    path = Path(path); path.mkdir(parents=True, exist_ok=True)
    sd = convert_state_dict_to_diffusers(get_peft_model_state_dict(unet))
    StableDiffusionInpaintPipeline.save_lora_weights(str(path), unet_lora_layers=sd)      # writes keys with the 'unet.' prefix

unet.train(); step, micro, hist, t0 = 0, 0, [], time.time()
it = iter(dl); run_loss = 0.0
while step < STEPS:
    b = next(it)
    x = b["pixel"].to(dev, torch.float16)
    with torch.no_grad():
        lat = vae.encode(x).latent_dist.sample() * vae.config.scaling_factor
        mlat = vae.encode(torch.zeros_like(x)).latent_dist.sample() * vae.config.scaling_factor   # nothing kept: paint the whole mural
    mask_l = torch.ones_like(lat[:, :1])                                                          # mask = whole image
    noise = torch.randn_like(lat); t = torch.randint(0, sched.config.num_train_timesteps, (lat.shape[0],), device=dev).long()
    noisy = sched.add_noise(lat, noise, t)
    emb = torch.cat([emb_null if d else emb_cap for d in b["drop"]], 0).to(torch.float16)
    with torch.autocast("cuda", dtype=torch.float16):
        pred = unet(torch.cat([noisy, mask_l, mlat], 1), t, encoder_hidden_states=emb).sample
    mse = (pred.float() - noise.float()) ** 2
    loss = mse.mean() / ACCUM
    scaler.scale(loss).backward(); run_loss += loss.item(); micro += 1
    if micro % ACCUM == 0:
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
        hist.append(run_loss); run_loss = 0.0
        if step % 25 == 0: print(f"step {step}/{STEPS}  loss(avg25) {np.mean(hist[-25:]):.4f}  {(time.time()-t0)/60:.1f} min", flush=True)
        if step % SAVE_EVERY == 0: save_lora(OUT / f"ckpt-{step}"); json.dump(hist, open(OUT/"loss_history.json", "w"))
save_lora(OUT / "final"); json.dump(hist, open(OUT/"loss_history.json", "w"))
import shutil; shutil.copy(OUT/"final"/"pytorch_lora_weights.safetensors", OUT/"dunhuang_inpaint_lora.safetensors")
print("done in", round((time.time()-t0)/60, 1), "min")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7,3)); plt.plot(np.convolve(hist, np.ones(25)/25, mode="valid")); plt.title("training loss (25-step average)"); plt.savefig(OUT/"loss.png", dpi=120); plt.show()
from safetensors import safe_open
with safe_open(str(OUT/"dunhuang_inpaint_lora.safetensors"), "pt") as f: ks = list(f.keys())
print(len(ks), "tensors; all start with 'unet.':", all(k.startswith("unet.") for k in ks)); print(ks[:2])

In [ ]:
# ---- quick visual check: murals painted from the prompt, base inpainting UNet vs + mural LoRA ----
del unet, opt; gc.collect(); torch.cuda.empty_cache()
pipe = StableDiffusionInpaintPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16, safety_checker=None).to(dev)
pipe.set_progress_bar_config(disable=True)
full = Image.fromarray(np.full((512, 512), 255, np.uint8)); rows = {}
for name in ["base", "lora"]:
    if name == "lora": pipe.load_lora_weights(str(OUT/"final"), weight_name="pytorch_lora_weights.safetensors"); pipe.fuse_lora(lora_scale=1.0)
    rows[name] = [np.asarray(pipe(prompt=CAPTION, image=Image.new("RGB", (512, 512)), mask_image=full, num_inference_steps=25,
                                  guidance_scale=6.0, generator=torch.Generator(dev).manual_seed(s)).images[0]) for s in range(4)]
fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for j in range(4): ax[0, j].imshow(rows["base"][j]); ax[1, j].imshow(rows["lora"][j])
for a, t in zip(ax[:, 0], ["base SD inpainting", "+ mural LoRA"]): a.set_ylabel(t)
for a in ax.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig(OUT/"base_vs_lora.png", dpi=110); plt.show()


**Next:** download `lora_inpaint/dunhuang_inpaint_lora.safetensors` and `lora_inpaint/base_vs_lora.png` and copy them into `models/`.